# Visualize motion path in recording playback

<video controls src="./assets/path_seeking_recording.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

Set up the server with playback of the universe:

In [2]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: trajectory seeker")

Import the jupyter utilities for drawing + interaction:

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Define a FrameListener to constantly update the ghost visuals as the nanotube moves during playback:

In [4]:
from nanover.utilities.transforms import StructureAlignment
from nanover.jupyter.alignment_transform import AlignmentTransform

NANOTUBE_ATOMS = range(0, 60)
alignment = StructureAlignment.from_atom_group(universe.atoms[NANOTUBE_ATOMS])

alignment_update = AlignmentTransform.from_runner(imd_runner)
alignment_update.config(key="nanotube", alignment=alignment)
alignment_update.start()

Calculate methane centroids in each frame and transform them into a common reference frame:

In [5]:
import numpy as np

METHANE_ATOMS = range(60, 65)

METHANE_CENTROIDS = []
for i, _ in enumerate(universe.trajectory):
    positions = universe.atoms.positions[METHANE_ATOMS] / 10  # angstrom -> nm
    centroid = np.mean(positions, axis=0)
    transform_i = alignment.calculate_transform_to_universe(universe)
    METHANE_CENTROIDS.append(transform_i.point_local_to_parent(centroid))

Add a line to the scene that shows the path of the centroid over time:

In [6]:
utilities.objects.update_line("path", positions=METHANE_CENTROIDS, size=0.01, parent="nanotube")

Define a mode that shows the nearest point on the centroid path as the user controllers hover close to it, and seeks the recording when they click:

In [7]:
from nanover.jupyter import Mode
from nanover.utilities.intersection import closest_point_on_polyline


class SeekLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        pos_world = cursor["position"]
        pos_scene = utilities.scene_transform.point_parent_to_local(pos_world)
        pos_align = alignment.calculate_transform_to_universe(universe).point_local_to_parent(pos_scene)

        result = closest_point_on_polyline(METHANE_CENTROIDS, pos_align)
        utilities.notify_all(f"seek to frame #{result.index}")
        simulation.seek_to_frame_index(result.index)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        pos_world = cursor["position"]
        pos_scene = utilities.scene_transform.point_parent_to_local(pos_world)
        pos_align = alignment.calculate_transform_to_universe(universe).point_local_to_parent(pos_scene)

        result = closest_point_on_polyline(METHANE_CENTROIDS, pos_align)

        if result.distance < 0.05:
            utilities.objects.update_shape(f"cursor.{key}", position=result.point, size=0.02, parent="nanotube")
        else:
            utilities.objects.remove_shape(f"cursor.{key}")


utilities.use_interaction_modes()
utilities.add_interaction_mode(SeekLineMode, "seek", icon="⏩")

Start playback running:

In [8]:
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
